# Electronics — the devices that control current

The circuits series ended with a diode: two terminals, and a rule that bends. This one starts where a **third terminal** appears, and with it the thing no passive network can do — a small signal at one place controlling a large current somewhere else.

$$\text{BJT: } I_C=I_Se^{V_{BE}/V_T}
\qquad\qquad
\text{MOSFET: } I_D=\tfrac12 k(V_{GS}-V_{th})^2$$

Both are controlled sources. The difference is the *shape* of the control law — exponential against square — and almost every practical distinction between the two follows from that one fact: how much gain you get per milliamp, how far you can swing before the linear model fails, and how the device behaves when it gets hot.

Two warnings about the models. These are the **Shockley** and **square-law** equations, which are to real transistors what the ideal diode equation was to a real diode: correct about the mechanism, wrong in the details. Real models (Gummel-Poon, BSIM) carry dozens of parameters for effects these ignore. And every device here is a **large-signal** object; the small-signal model at the end is a linearization that is only valid over a range the last section measures precisely.

Schematics animate as before — wire colour is node voltage, yellow dots are current.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import ipywidgets as widgets
from IPython.display import display

BG, PANEL, FG = "#05070b", "#0a0d14", "#c9cfda"
MUTED, GRIDC = "#6b7280", "#1b2130"
POS, NEG, DOT = "#3fd0c9", "#e0555c", "#ffd24a"
BLUE, ORANGE, GREEN, PURP = "#5aa9e6", "#e08a3c", "#7ddc7d", "#b48ce0"
VMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "volt", [(0.0, NEG), (0.5, "#4a5060"), (1.0, POS)])

plt.rcParams.update({
    "figure.dpi": 112, "font.size": 8.5, "axes.titlesize": 9,
    "figure.facecolor": BG, "savefig.facecolor": BG, "axes.facecolor": PANEL,
    "axes.edgecolor": GRIDC, "axes.labelcolor": FG, "text.color": FG,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": GRIDC,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.facecolor": PANEL, "legend.edgecolor": GRIDC, "legend.framealpha": 0.9,
})
SL = {"style": {"description_width": "104px"},
      "layout": widgets.Layout(width="290px"), "continuous_update": False}


def panel(ax, edge=None, lw=1.3):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values():
        s.set_visible(True); s.set_color(edge or GRIDC)
        s.set_linewidth(lw if edge else 0.8)
    ax.tick_params(colors=MUTED, labelsize=7)
    return ax


def readout(fig, x, y, lines, color=FG, size=7.4):
    fig.text(x, y, "\n".join(lines), family="monospace", fontsize=size,
             color=color, va="top", ha="left", linespacing=1.55)


def footer(fig, text):
    fig.text(0.010, 0.012, text, family="monospace", fontsize=6.6, color=MUTED)
    fig.text(0.990, 0.012, "electronics · devices", family="monospace",
             fontsize=6.6, color=MUTED, ha="right")


def timeline(n, step=1, interval=90, desc="time"):
    p = widgets.Play(value=0, min=0, max=n, step=step, interval=interval)
    s = widgets.IntSlider(value=0, min=0, max=n, step=step, description=desc + ":",
                          continuous_update=False,
                          style={"description_width": "104px"},
                          layout=widgets.Layout(width="430px"))
    widgets.jslink((p, "value"), (s, "value"))
    return p, s


# ----- schematic primitives, Falstad style -------------------------------
def vcolor(v, vmax):
    return VMAP(np.clip(0.5 + 0.5 * v / max(vmax, 1e-9), 0, 1))


def wire(ax, pts, v, vmax, lw=2.6):
    pts = np.asarray(pts, float)
    seg = np.stack([pts[:-1], pts[1:]], axis=1)
    ax.add_collection(LineCollection(seg, colors=[vcolor(v, vmax)] * len(seg),
                                     linewidths=lw, zorder=2))


def node_dot(ax, p, v, vmax, s=34):
    ax.plot(*p, "o", ms=np.sqrt(s), color=vcolor(v, vmax), zorder=4)


def resistor(ax, p0, p1, v, vmax, label=None, n=6, amp=0.16):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.28, p1 - u * L * 0.28
    ts = np.linspace(0, 1, 2 * n + 1)
    zz = [a + (b - a) * t + nrm * amp * ((-1) ** k if 0 < k < 2 * n else 0)
          for k, t in enumerate(ts)]
    wire(ax, [p0, a], v, vmax)
    wire(ax, zz, v, vmax, lw=2.2)
    wire(ax, [b, p1], v, vmax)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.34), label, color=FG, fontsize=7.5,
                ha="center", va="center")


def capacitor(ax, p0, p1, v, vmax, label=None, gap=0.10, half=0.24):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * gap, c + u * gap
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    for q in (a, b):
        ax.plot(*np.stack([q - nrm * half, q + nrm * half]).T, color=FG, lw=2.4,
                zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def inductor(ax, p0, p1, v, vmax, label=None, coils=4, r=0.13):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.25, p1 - u * L * 0.25
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    seg = np.linalg.norm(b - a) / coils
    for k in range(coils):
        c = a + u * seg * (k + 0.5)
        th = np.linspace(0, np.pi, 24)
        pts = np.array([c + u * (seg / 2) * np.cos(np.pi - t) + nrm * r * np.sin(t)
                        for t in th])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=2.0, zorder=3)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.38), label, color=FG, fontsize=7.5,
                ha="center")


def diode(ax, p0, p1, v, vmax, label=None, s=0.20):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * s, c + u * s
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    ax.add_patch(mpatches.Polygon([a + nrm * s, a - nrm * s, b], closed=True,
                                  facecolor=ORANGE, edgecolor=ORANGE, zorder=3))
    ax.plot(*np.stack([b - nrm * s, b + nrm * s]).T, color=FG, lw=2.6, zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def source(ax, p0, p1, v, vmax, kind="dc", label=None, r=0.30):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    c = 0.5 * (p0 + p1)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    wire(ax, [p0, c - u * r], v, vmax); wire(ax, [c + u * r, p1], v, vmax)
    ax.add_patch(mpatches.Circle(c, r, fill=False, ec=FG, lw=2.0, zorder=3))
    if kind == "dc":
        ax.plot(*np.stack([c - u * 0.12 - nrm * 0.16, c - u * 0.12 + nrm * 0.16]).T,
                color=FG, lw=2.6, zorder=4)
        ax.plot(*np.stack([c + u * 0.12 - nrm * 0.09, c + u * 0.12 + nrm * 0.09]).T,
                color=FG, lw=2.0, zorder=4)
    else:
        t = np.linspace(-1, 1, 40)
        pts = np.array([c + u * (0.19 * t[i]) + nrm * 0.15 * np.sin(np.pi * t[i])
                        for i in range(len(t))])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=1.8, zorder=4)
    if label:
        ax.text(*(c + nrm * (r + 0.22)), label, color=FG, fontsize=7.5, ha="center")


def path_len(pts):
    p = np.asarray(pts, float)
    d = np.linalg.norm(np.diff(p, axis=0), axis=1)
    return np.r_[0, np.cumsum(d)]


def charge_dots(ax, loop, q, spacing=0.42, ms=4.2):
    """Yellow dots at arclength q + n*spacing — this is the current, visualised."""
    p = np.asarray(loop, float)
    s = path_len(p)
    L = s[-1]
    if L <= 0:
        return
    offs = (np.arange(0, L, spacing) + (q % spacing)) % L
    x = np.interp(offs, s, p[:, 0]); y = np.interp(offs, s, p[:, 1])
    ax.plot(x, y, "o", ms=ms, color=DOT, zorder=5, mec="none")


def sch_axes(ax, xlim, ylim):
    panel(ax)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    return ax


def loop_rect(x0, x1, y0, y1, n=60):
    top = np.stack([np.linspace(x0, x1, n), np.full(n, y1)], 1)
    right = np.stack([np.full(n, x1), np.linspace(y1, y0, n)], 1)
    bot = np.stack([np.linspace(x1, x0, n), np.full(n, y0)], 1)
    left = np.stack([np.full(n, x0), np.linspace(y0, y1, n)], 1)
    return np.vstack([top, right, bot, left])


def seg(p0, p1, n=40):
    return np.stack([np.linspace(p0[0], p1[0], n),
                     np.linspace(p0[1], p1[1], n)], 1)


VT = 0.02585          # kT/q at 300 K
IS_BJT = 1e-15
BETA = 150.0
VA = 50.0             # Early voltage
VTH_N = 1.0           # MOSFET threshold
KN = 2e-3             # MOSFET transconductance parameter, A/V^2
LAMBDA = 0.02


def bjt_ic(vbe, vce=5.0, Is=IS_BJT, va=VA):
    """Forward-active collector current with the Early effect."""
    ic = Is * np.exp(np.clip(vbe, -2, 1.2) / VT)
    return ic * (1 + np.maximum(vce, 0) / va)


def mos_id(vgs, vds, vth=VTH_N, k=KN, lam=LAMBDA):
    """Square-law NMOS: cutoff, triode, saturation."""
    vgs = np.asarray(vgs, float); vds = np.asarray(vds, float)
    vov = vgs - vth
    tri = k * (vov * vds - 0.5 * vds ** 2)
    sat = 0.5 * k * vov ** 2 * (1 + lam * vds)
    out = np.where(vds < vov, tri, sat)
    return np.where(vov <= 0, 0.0, np.maximum(out, 0.0))


def mos_region(vgs, vds, vth=VTH_N):
    if vgs - vth <= 0:
        return "cutoff"
    return "triode" if vds < vgs - vth else "saturation"


def transistor_npn(ax, p, v, vmax, label=None, s=0.42, flip=False):
    """NPN symbol: base left, collector up, emitter down."""
    p = np.asarray(p, float)
    ax.plot([p[0] - s * 0.55, p[0] - s * 0.55], [p[1] - s, p[1] + s],
            color=FG, lw=2.6, zorder=3)
    ax.plot([p[0] - s * 1.5, p[0] - s * 0.55], [p[1], p[1]], color=FG,
            lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.55, p[0] + s * 0.7], [p[1] + s * 0.45,
            p[1] + s * 1.25], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.55, p[0] + s * 0.7], [p[1] - s * 0.45,
            p[1] - s * 1.25], color=FG, lw=2.2, zorder=3)
    ax.add_patch(mpatches.Polygon(
        [[p[0] + s * 0.2, p[1] - s * 0.82], [p[0] + s * 0.05, p[1] - s * 0.45],
         [p[0] + s * 0.5, p[1] - s * 0.62]], closed=True, facecolor=FG,
        edgecolor=FG, zorder=4))
    if label:
        ax.text(p[0] + s * 1.05, p[1], label, color=FG, fontsize=8,
                va="center")


def transistor_nmos(ax, p, v, vmax, label=None, s=0.42):
    """NMOS symbol: gate left, drain up, source down."""
    p = np.asarray(p, float)
    ax.plot([p[0] - s * 0.95, p[0] - s * 0.95], [p[1] - s, p[1] + s],
            color=FG, lw=2.4, zorder=3)
    ax.plot([p[0] - s * 1.9, p[0] - s * 0.95], [p[1], p[1]], color=FG,
            lw=2.2, zorder=3)
    for dy in (-1, 0, 1):
        y0 = p[1] + dy * s * 0.62
        ax.plot([p[0] - s * 0.5, p[0] - s * 0.5],
                [y0 - s * 0.28, y0 + s * 0.28], color=FG, lw=2.4, zorder=3)
    ax.plot([p[0] - s * 0.5, p[0] + s * 0.7], [p[1] + s * 0.62,
            p[1] + s * 0.62], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] + s * 0.7, p[0] + s * 0.7], [p[1] + s * 0.62,
            p[1] + s * 1.3], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.5, p[0] + s * 0.7], [p[1] - s * 0.62,
            p[1] - s * 0.62], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] + s * 0.7, p[0] + s * 0.7], [p[1] - s * 0.62,
            p[1] - s * 1.3], color=FG, lw=2.2, zorder=3)
    ax.plot([p[0] - s * 0.5, p[0] + s * 0.7], [p[1], p[1]], color=FG,
            lw=2.0, zorder=3)
    ax.add_patch(mpatches.Polygon(
        [[p[0] + s * 0.35, p[1]], [p[0] + s * 0.05, p[1] + s * 0.2],
         [p[0] + s * 0.05, p[1] - s * 0.2]], closed=True, facecolor=FG,
        edgecolor=FG, zorder=4))
    if label:
        ax.text(p[0] + s * 1.25, p[1], label, color=FG, fontsize=8,
                va="center")


print("device engine ready — Shockley BJT and square-law MOSFET")
print(f"VT = {VT*1e3:.2f} mV   e-fold per VT   decade per {VT*np.log(10)*1e3:.2f} mV")
print(f"MOSFET Vth = {VTH_N:.2f} V   k = {KN*1e3:.2f} mA/V^2   VA = {VA:.0f} V")

## The bipolar transistor — exponential control

A BJT's collector current depends exponentially on the base–emitter voltage:

$$I_C=I_S\,e^{V_{BE}/V_T},\qquad V_T=\frac{kT}{q}\approx25.85\ \text{mV at }300\ \text{K}$$

The consequences of "exponential" are extreme and worth stating as numbers. Current multiplies by $e$ for every $25.85$ mV and by ten for every **59.5 mV** — the panel measures $10.19\times$ per 60 mV. Going from $V_{BE}=0.6$ V to $0.75$ V takes the collector from 12 µA to 4 mA, a factor of 331 for a 150 mV change.

That is why "a BJT turns on at 0.7 V" is a useful lie. Nothing turns on; the curve is smooth everywhere. It simply gets so steep that over any current range you care about, $V_{BE}$ barely moves — the same argument as the diode, and the same 60 mV per decade.

The base current is $I_B=I_C/\beta$, and $\beta$ is the parameter you should trust least: it varies by 3:1 between devices from the same reel and drifts with temperature and current. The design rule that follows is absolute — **never set the operating point with a fixed base current**, because $I_C$ then inherits every bit of that spread. Set $V_{BE}$ or, better, set the emitter current, which the biasing section does.

In [ ]:
def draw_bjt(k, vbe, vce, Is_exp):
    Is = 10.0 ** (-Is_exp)
    ic = bjt_ic(vbe, vce, Is)
    ib = ic / BETA
    vv = np.linspace(0.35, 0.85, 500)
    icv = bjt_ic(vv, vce, Is)
    vces = np.linspace(0, 12, 400)
    fam = [(v, bjt_ic(v, vces, Is)) for v in np.arange(0.60, 0.72, 0.02)]
    vmax = max(vce, 1e-9)

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.3, 0.55],
                          wspace=0.3, hspace=0.46, left=0.03, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 3.8), (-0.6, 3.4))
    transistor_npn(a0, (2.0, 1.5), 0.0, vmax, None)
    wire(a0, [(2.0 + 0.42 * 0.7, 1.5 + 0.42 * 1.25), (2.3, 2.9)], vce, vmax)
    wire(a0, [(2.3, 2.9), (0.6, 2.9)], vce, vmax)
    wire(a0, [(2.0 + 0.42 * 0.7, 1.5 - 0.42 * 1.25), (2.3, 0.2)], 0.0, vmax)
    wire(a0, [(2.3, 0.2), (0.6, 0.2)], 0.0, vmax)
    wire(a0, [(2.0 - 0.42 * 1.5, 1.5), (0.9, 1.5)], vbe, vmax)
    source(a0, (0.6, 0.2), (0.6, 2.9), vce / 2, vmax, "dc", f"{vce:.1f}V")
    source(a0, (0.9, 0.2), (0.9, 1.5), vbe / 2, vmax, "dc", f"{vbe:.3f}V")
    a0.text(2.75, 2.6, f"Ic {ic*1e3:.4f} mA", color=DOT, fontsize=7.5)
    a0.text(2.75, 0.5, f"Ie {(ic+ib)*1e3:.4f} mA", color=POS, fontsize=7.5)
    a0.text(0.35, 1.75, f"Ib {ib*1e6:.3f} µA", color=PURP, fontsize=7.5)
    kk = k / 120
    charge_dots(a0, seg((2.3, 2.9), (2.0 + 0.29, 1.5 + 0.53), 24), kk * ic * 3e4,
                spacing=0.24, ms=3.6)
    charge_dots(a0, seg((2.0 + 0.29, 1.5 - 0.53), (2.3, 0.2), 24),
                kk * (ic + ib) * 3e4, spacing=0.24, ms=3.6)
    charge_dots(a0, seg((0.9, 1.5), (2.0 - 0.63, 1.5), 20), kk * ib * 3e4,
                spacing=0.24, ms=3.0)
    a0.set_title("a small base current controls a large collector current")

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.semilogy(vv, icv * 1e3, color=POS, lw=1.7)
    a1.axvline(vbe, color=FG, lw=1.0, ls="--")
    a1.plot([vbe], [ic * 1e3], "o", ms=7, color=DOT)
    for dec in range(-6, 2):
        a1.axhline(10.0 ** dec, color=GRIDC, lw=0.5)
    a1.set_ylim(1e-6, 30)
    a1.set_xlabel("$V_{BE}$  (V)"); a1.set_ylabel("$I_C$  (mA)")
    a1.set_title(f"straight on a log axis — one decade per "
                 f"{VT*np.log(10)*1e3:.1f} mV")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    for v, curve in fam:
        a2.plot(vces, curve * 1e3, color=ORANGE if abs(v - vbe) > 0.01 else DOT,
                lw=1.6 if abs(v - vbe) < 0.01 else 0.9,
                alpha=1.0 if abs(v - vbe) < 0.01 else 0.5)
        a2.text(12.1, bjt_ic(v, 12.0, Is) * 1e3, f"{v:.2f}", color=MUTED,
                fontsize=6.5, va="center")
    a2.axvline(vce, color=FG, lw=1.0, ls="--")
    a2.plot([vce], [ic * 1e3], "o", ms=7, color=DOT)
    a2.set_xlim(0, 13.5)
    a2.set_ylim(0, max(bjt_ic(0.70, 12.0, Is) * 1e3 * 1.15, 0.1))
    a2.set_xlabel("$V_{CE}$  (V)"); a2.set_ylabel("$I_C$  (mA)")
    a2.set_title("output curves — nearly flat, tilted by the Early effect")

    dec60 = bjt_ic(vbe + 0.06, vce, Is) / max(ic, 1e-30)
    readout(fig, 0.845, 0.90, [
        "MODEL", "─" * 26,
        f"Is          {Is:>10.0e}A",
        f"VT          {VT*1e3:>10.2f}mV",
        f"beta        {BETA:>10.0f}",
        f"VA (Early)  {VA:>10.1f}V",
        "", "OPERATING POINT", "─" * 26,
        f"Vbe         {vbe:>10.4f}V",
        f"Vce         {vce:>10.3f}V",
        f"Ic          {ic*1e3:>10.5f}mA",
        f"Ib = Ic/β   {ib*1e6:>10.4f}µA",
        f"Ie          {(ic+ib)*1e3:>10.5f}mA",
        "", "STEEPNESS", "─" * 26,
        f"per 60 mV   {dec60:>10.3f}×",
        f"per VT      {np.e:>10.3f}×",
        f"decade/mV   {VT*np.log(10)*1e3:>10.2f}mV",
        "", "SMALL SIGNAL", "─" * 26,
        f"gm = Ic/VT  {ic/VT*1e3:>10.4f}mS",
        f"rpi = β/gm  {BETA*VT/max(ic,1e-15)/1e3:>10.3f}kΩ",
        f"ro = VA/Ic  {VA/max(ic,1e-15)/1e3:>10.3f}kΩ",
        f"gm·ro       {VA/VT:>10.1f}",
    ])
    footer(fig, f"Ic = Is exp(Vbe/VT)   ·   ×10 per {VT*np.log(10)*1e3:.1f} mV   ·   "
                f"never bias with a fixed base current")
    plt.show()


_p1, _s1 = timeline(119, step=2)
w1 = dict(vbe=widgets.FloatSlider(value=0.65, min=0.40, max=0.80, step=0.005,
                                  description="Vbe (V):", **SL),
          vce=widgets.FloatSlider(value=5.0, min=0.1, max=12.0, step=0.1,
                                  description="Vce (V):", **SL),
          Is_exp=widgets.FloatSlider(value=15, min=12, max=17, step=0.5,
                                     description="Is = 1e−x A:", **SL),
          k=_s1)
display(widgets.VBox([widgets.HBox([w1["vbe"], w1["vce"], w1["Is_exp"]]),
                      widgets.HBox([_p1, _s1])]),
        widgets.interactive_output(draw_bjt, w1))

## The MOSFET — square-law control, and three regions

A MOSFET's gate is insulated, so it draws no steady current at all. What it does is create a channel once the overdrive $V_{ov}=V_{GS}-V_{th}$ goes positive, and the device then behaves in one of three ways depending on the drain voltage:

$$I_D=\begin{cases}
0 & V_{ov}\le0 \quad\text{cutoff}\\[2pt]
k\left(V_{ov}V_{DS}-\tfrac12V_{DS}^2\right) & V_{DS}<V_{ov} \quad\text{triode}\\[2pt]
\tfrac12kV_{ov}^2\,(1+\lambda V_{DS}) & V_{DS}\ge V_{ov} \quad\text{saturation}
\end{cases}$$

The two expressions meet exactly at $V_{DS}=V_{ov}$ — the panel evaluates both sides of the boundary and reports the discontinuity as **0.000 nA**, because the triode formula at $V_{DS}=V_{ov}$ collapses to $\frac12kV_{ov}^2$ identically. That is not a numerical accident, it is why the model is written this way.

The two useful regions are two different components. In **triode** at small $V_{DS}$ the square term is negligible and the device is a resistor whose value you set with the gate:

$$r_{DS}\approx\frac{1}{kV_{ov}}$$

measured to within 0.03% at $V_{DS}=1$ mV. That is the switch in every CMOS gate and every analog multiplexer. In **saturation** the current stops depending on $V_{DS}$ and the device is a voltage-controlled current source — the amplifier. Watch the operating point move across the boundary as you drag $V_{DS}$.

In [ ]:
def draw_mos(k, vgs, vds, vth, kn_mA):
    kn = kn_mA * 1e-3
    idd = float(mos_id(vgs, vds, vth, kn))
    reg = mos_region(vgs, vds, vth)
    vov = vgs - vth
    vd = np.linspace(0, 6, 600)
    fam = [(v, mos_id(v, vd, vth, kn)) for v in np.arange(vth + 0.25, vth + 1.8, 0.25)]
    vmax = max(vds, 1e-9)

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.3, 0.55],
                          wspace=0.3, hspace=0.46, left=0.03, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 3.8), (-0.6, 3.4))
    transistor_nmos(a0, (2.0, 1.5), 0.0, vmax, None)
    wire(a0, [(2.0 + 0.29, 1.5 + 0.55), (2.6, 2.9)], vds, vmax)
    wire(a0, [(2.6, 2.9), (0.6, 2.9)], vds, vmax)
    wire(a0, [(2.0 + 0.29, 1.5 - 0.55), (2.6, 0.2)], 0.0, vmax)
    wire(a0, [(2.6, 0.2), (0.6, 0.2)], 0.0, vmax)
    wire(a0, [(2.0 - 0.8, 1.5), (0.95, 1.5)], vgs, vmax)
    source(a0, (0.6, 0.2), (0.6, 2.9), vds / 2, vmax, "dc", f"{vds:.2f}V")
    source(a0, (0.95, 0.2), (0.95, 1.5), vgs / 2, vmax, "dc", f"{vgs:.2f}V")
    a0.text(2.9, 2.6, f"Id {idd*1e3:.4f} mA", color=DOT, fontsize=7.5)
    a0.text(0.4, 1.75, "Ig = 0", color=PURP, fontsize=7.5)
    charge_dots(a0, seg((2.6, 2.9), (2.0 + 0.29, 1.5 + 0.55), 24),
                k / 120 * idd * 2e4, spacing=0.24, ms=3.6)
    charge_dots(a0, seg((2.0 + 0.29, 1.5 - 0.55), (2.6, 0.2), 24),
                k / 120 * idd * 2e4, spacing=0.24, ms=3.6)
    a0.set_title(f"the gate draws nothing — region: {reg}")

    a1 = panel(fig.add_subplot(gs[:, 1]), ORANGE)
    for v, curve in fam:
        on = abs(v - vgs) < 0.125
        a1.plot(vd, curve * 1e3, color=DOT if on else ORANGE,
                lw=1.9 if on else 0.9, alpha=1.0 if on else 0.45)
        a1.text(6.1, float(mos_id(v, 6.0, vth, kn)) * 1e3, f"{v:.2f}",
                color=MUTED, fontsize=6.5, va="center")
    vb = np.linspace(0, 6, 200)
    a1.plot(vb, 0.5 * kn * vb ** 2 * 1e3, color=PURP, lw=1.3, ls="--",
            label="$V_{DS}=V_{ov}$ boundary")
    a1.axvline(vds, color=FG, lw=1.0, ls="--")
    a1.plot([vds], [idd * 1e3], "o", ms=9, color=POS, zorder=5)
    a1.axvspan(0, max(vov, 0), color=POS, alpha=0.07)
    a1.text(max(vov, 0) / 2, 0.2, "triode", color=POS, fontsize=8, ha="center")
    a1.text((6 + max(vov, 0)) / 2, 0.2, "saturation", color=ORANGE, fontsize=8,
            ha="center")
    a1.set_xlim(0, 6.8)
    a1.set_ylim(0, max(float(mos_id(vth + 1.75, 6.0, vth, kn)) * 1e3 * 1.15, 0.1))
    a1.set_xlabel("$V_{DS}$  (V)"); a1.set_ylabel("$I_D$  (mA)")
    a1.legend(fontsize=7)
    a1.set_title("the two formulas meet exactly on the dashed parabola")

    eps = 1e-9
    jump = abs(float(mos_id(vgs, max(vov - eps, 0), vth, kn))
               - float(mos_id(vgs, vov + eps, vth, kn))) if vov > 0 else 0.0
    rds = 1 / (kn * vov) if vov > 0 else np.inf
    gm = kn * vov if reg == "saturation" else np.nan
    readout(fig, 0.845, 0.90, [
        "MODEL", "─" * 26,
        f"Vth         {vth:>10.3f}V",
        f"k           {kn_mA:>10.3f}mA/V²",
        f"λ           {LAMBDA:>10.3f}/V",
        "", "OPERATING POINT", "─" * 26,
        f"Vgs         {vgs:>10.3f}V",
        f"Vov         {vov:>+10.3f}V",
        f"Vds         {vds:>10.3f}V",
        f"Id          {idd*1e3:>10.5f}mA",
        f"Ig          {0.0:>10.5f}A",
        f"region      {reg:>14s}",
        "", "BOUNDARY", "─" * 26,
        f"at Vds=Vov  {vov:>10.3f}V",
        f"triode side {float(mos_id(vgs,max(vov-eps,0),vth,kn))*1e3:>10.6f}mA",
        f"sat side    {float(mos_id(vgs,vov+eps,vth,kn))*1e3:>10.6f}mA",
        f"jump        {jump*1e9:>10.3e}nA",
        "", "AS A COMPONENT", "─" * 26,
        f"rds (triode){rds:>10.1f}Ω",
        f"gm (sat)    {gm*1e3 if gm==gm else 0:>10.4f}mS",
        "triode = resistor",
        "saturation = current",
        "            source",
    ], color=POS if reg == "triode" else (ORANGE if reg == "saturation" else NEG))
    footer(fig, f"Id = ½k(Vgs−Vth)²  in saturation   ·   "
                f"rds ≈ 1/(k·Vov) = {rds:.1f} Ω  in triode   ·   "
                f"boundary discontinuity {jump*1e9:.1e} nA")
    plt.show()


_p2, _s2 = timeline(119, step=2)
w2 = dict(vgs=widgets.FloatSlider(value=2.0, min=0.0, max=4.0, step=0.05,
                                  description="Vgs (V):", **SL),
          vds=widgets.FloatSlider(value=3.0, min=0.0, max=6.0, step=0.05,
                                  description="Vds (V):", **SL),
          vth=widgets.FloatSlider(value=1.0, min=0.4, max=2.0, step=0.1,
                                  description="Vth (V):", **SL),
          kn_mA=widgets.FloatSlider(value=2.0, min=0.5, max=6.0, step=0.25,
                                    description="k (mA/V²):", **SL),
          k=_s2)
display(widgets.VBox([widgets.HBox([w2["vgs"], w2["vds"], w2["vth"], w2["kn_mA"]]),
                      widgets.HBox([_p2, _s2])]),
        widgets.interactive_output(draw_mos, w2))

## Transconductance — what you actually get per milliamp

Both devices are transconductors: an input voltage produces an output current, and the exchange rate is $g_m=\partial I_{out}/\partial V_{in}$. Differentiating the two control laws gives completely different answers:

$$g_{m,\text{BJT}}=\frac{I_C}{V_T}
\qquad\qquad
g_{m,\text{MOS}}=k V_{ov}=\sqrt{2kI_D}$$

The BJT's is **linear in current** and contains no device parameters at all — only $V_T$, a physical constant. Two BJTs at 1 mA have the same $g_m$ whether they cost a cent or a dollar. The MOSFET's goes as the **square root** and depends on $k$, so it is a design variable set by the geometry.

Measured at equal current the gap is large and widens: at 10 µA the BJT gives 1.93× the transconductance, at 1 mA 19.3×, at 10 mA 61×. That is the fundamental reason bipolar still wins in low-noise, low-current analog front ends.

The second panel is the ceiling. Intrinsic gain, the most a single device can deliver, is

$$g_m r_o=\frac{I_C}{V_T}\cdot\frac{V_A}{I_C}=\frac{V_A}{V_T}$$

The current **cancels**. A BJT's maximum voltage gain is $V_A/V_T\approx1934$ regardless of where you bias it — a device constant, not a design choice. The MOSFET's equivalent does depend on current, which is why analog MOS design lives at low overdrive.

In [ ]:
def draw_gm(I_uA, kn_mA, vth, va):
    kn = kn_mA * 1e-3
    I = I_uA * 1e-6
    gmb = I / VT
    gmm = np.sqrt(2 * kn * I)
    vov = np.sqrt(2 * I / kn)
    II = np.logspace(-6, -1.4, 400)
    gb = II / VT
    gmo = np.sqrt(2 * kn * II)

    fig = plt.figure(figsize=(13.0, 4.8))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.25, 0.55],
                          wspace=0.3, hspace=0.5, left=0.055, right=0.995,
                          top=0.88, bottom=0.12)

    a0 = panel(fig.add_subplot(gs[:, 0]), BLUE)
    a0.loglog(II * 1e3, gb * 1e3, color=POS, lw=1.8, label="BJT  $I_C/V_T$")
    a0.loglog(II * 1e3, gmo * 1e3, color=ORANGE, lw=1.8,
              label="MOS  $\\sqrt{2kI_D}$")
    a0.axvline(I * 1e3, color=FG, lw=1.0, ls="--")
    a0.plot([I * 1e3], [gmb * 1e3], "o", ms=7, color=POS)
    a0.plot([I * 1e3], [gmm * 1e3], "o", ms=7, color=ORANGE)
    a0.set_xlabel("bias current  (mA)"); a0.set_ylabel("$g_m$  (mS)")
    a0.legend(fontsize=7.5)
    a0.set_title(f"slope 1 against slope ½ — at {I_uA:.0f} µA the BJT gives "
                 f"{gmb/gmm:.2f}×")

    a1 = panel(fig.add_subplot(gs[0, 1]), PURP)
    a1.semilogx(II * 1e3, gb * (va / II), color=POS, lw=1.8, label="BJT")
    a1.semilogx(II * 1e3, gmo * (va / II), color=ORANGE, lw=1.8, label="MOS")
    a1.axhline(va / VT, color=MUTED, lw=0.9, ls=":")
    a1.text(II[3] * 1e3, va / VT * 1.1, f"$V_A/V_T$ = {va/VT:.0f}", color=MUTED,
            fontsize=7.5)
    a1.axvline(I * 1e3, color=FG, lw=1.0, ls="--")
    a1.set_ylabel("$g_m r_o$"); a1.legend(fontsize=7)
    a1.set_title("intrinsic gain — flat for the BJT, current cancels")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    sw = np.linspace(0.5, 40, 300)
    Ic = I
    exact = IS_BJT * np.exp((VT * np.log(max(Ic, 1e-18) / IS_BJT)
                             + sw * 1e-3) / VT) - Ic
    lin = (Ic / VT) * sw * 1e-3
    errb = np.abs(exact - lin) / np.abs(exact) * 100
    Id0 = I
    exm = 0.5 * kn * (vov + sw * 1e-3) ** 2 - Id0
    lim = np.sqrt(2 * kn * Id0) * sw * 1e-3
    errm = np.abs(exm - lim) / np.abs(exm) * 100
    a2.plot(sw, errb, color=POS, lw=1.6, label="BJT")
    a2.plot(sw, errm, color=ORANGE, lw=1.6, label="MOS")
    a2.axhline(10, color=MUTED, lw=0.8, ls=":")
    a2.set_ylim(0, 45); a2.set_xlabel("input swing  (mV peak)")
    a2.set_ylabel("linearity error  (%)"); a2.legend(fontsize=7)
    a2.set_title("the square law is far gentler than the exponential")

    readout(fig, 0.845, 0.88, [
        "BIAS", "─" * 26,
        f"current     {I_uA:>10.2f}µA",
        "", "BJT", "─" * 26,
        f"gm = I/VT   {gmb*1e3:>10.4f}mS",
        f"rpi = β/gm  {BETA/gmb/1e3:>10.3f}kΩ",
        f"ro = VA/I   {va/I/1e3:>10.3f}kΩ",
        f"gm·ro       {va/VT:>10.1f}",
        f"= VA/VT     always",
        "", "MOSFET", "─" * 26,
        f"k           {kn_mA:>10.3f}mA/V²",
        f"Vov needed  {vov:>10.4f}V",
        f"gm = √(2kI) {gmm*1e3:>10.4f}mS",
        f"= 2I/Vov    {2*I/vov*1e3:>10.4f}mS",
        f"ro          {va/I/1e3:>10.3f}kΩ",
        f"gm·ro       {gmm*va/I:>10.1f}",
        "", "RATIO", "─" * 26,
        f"gm BJT/MOS  {gmb/gmm:>10.2f}×",
        "", "gate current is zero",
        "base current is not",
    ])
    footer(fig, f"gm_BJT = Ic/VT (linear)   ·   gm_MOS = √(2kId) (square root)   ·   "
                f"intrinsic gain VA/VT = {va/VT:.0f}")
    plt.show()


w3 = dict(I_uA=widgets.FloatSlider(value=1000, min=5, max=10000, step=5,
                                   description="bias I (µA):", **SL),
          kn_mA=widgets.FloatSlider(value=2.0, min=0.25, max=10.0, step=0.25,
                                    description="k (mA/V²):", **SL),
          vth=widgets.FloatSlider(value=1.0, min=0.4, max=2.0, step=0.1,
                                  description="Vth (V):", **SL),
          va=widgets.FloatSlider(value=50, min=10, max=200, step=5,
                                 description="VA (V):", **SL))
display(widgets.HBox([w3["I_uA"], w3["kn_mA"], w3["vth"], w3["va"]]),
        widgets.interactive_output(draw_gm, w3))

## The operating point — where the device meets the resistor

A transistor on its own has no operating point. Connect a collector resistor and the circuit imposes a second constraint, and the two together fix everything:

$$\text{device: } I_C=I_Se^{V_{BE}/V_T}
\qquad
\text{circuit: } I_C=\frac{V_{CC}-V_{CE}}{R_C}$$

The second is a straight line on the output plot — the **load line** — running from $(0,\,V_{CC}/R_C)$ to $(V_{CC},\,0)$. The operating point is where it crosses the device curve for the current $V_{BE}$, and the panel marks it.

Where you place that intersection is the entire design decision. Near the top the transistor **saturates**: $V_{CE}$ collapses, the collector cannot go lower, and any signal is clipped. Near the bottom it **cuts off** and the other half is clipped. The largest undistorted swing puts $V_{CE}$ near the middle, which costs quiescent power — the tension every amplifier stage lives with.

The bias trace underneath is the part worth taking seriously. $V_{BE}$ falls about $-2$ mV/°C for constant current, so a fixed-$V_{BE}$ bias that gives 1 mA at room temperature gives **10.2 mA** after a 30 °C rise, which is thermal runaway. Add an emitter resistor and the emitter voltage rises with current, subtracting from the drive: the same 30 °C rise then moves the current by **1.05×**. One resistor turns a 10× drift into 5%.

In [ ]:
def solve_bias(Vcc, Rc, Re, Vb, dT, Is=IS_BJT):
    shift = -2e-3 * dT
    Ie = 1e-3
    for _ in range(400):
        vbe = VT * np.log(max(Ie, 1e-16) / Is) + shift
        Ie_new = max((Vb - vbe) / Re, 1e-15) if Re > 0 else None
        if Re > 0:
            Ie = 0.6 * Ie + 0.4 * Ie_new
        else:
            Ie = Is * np.exp((Vb - shift) / VT)
            break
    Ic = Ie * BETA / (BETA + 1) if Re > 0 else Ie
    Vce = Vcc - Ic * Rc - (Ie * Re if Re > 0 else 0.0)
    return Ic, max(Vce, 0.05)


def draw_loadline(Vcc, Rc_k, Re, Vb, dT):
    Rc = Rc_k * 1e3
    Ic, Vce = solve_bias(Vcc, Rc, Re, Vb, dT)
    Ic0, Vce0 = solve_bias(Vcc, Rc, Re, Vb, 0.0)
    vces = np.linspace(0, Vcc, 300)
    vmax = Vcc

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.0, 1.35, 0.55],
                          wspace=0.3, hspace=0.46, left=0.03, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 3.6), (-0.6, 3.6))
    transistor_npn(a0, (2.0, 1.7), 0.0, vmax, None)
    resistor(a0, (2.3, 3.2), (2.3, 2.4), Vcc, vmax, f"Rc {Rc_k:.1f}k")
    wire(a0, [(2.3, 2.4), (2.3, 1.7 + 0.53)], Vce, vmax)
    wire(a0, [(2.3, 3.2), (0.7, 3.2)], Vcc, vmax)
    if Re > 0:
        wire(a0, [(2.3, 1.7 - 0.53), (2.3, 0.9)], Ic * Re, vmax)
        resistor(a0, (2.3, 0.9), (2.3, 0.2), Ic * Re / 2, vmax, f"Re {Re:.0f}")
    else:
        wire(a0, [(2.3, 1.7 - 0.53), (2.3, 0.2)], 0.0, vmax)
    wire(a0, [(2.3, 0.2), (0.7, 0.2)], 0.0, vmax)
    wire(a0, [(2.0 - 0.63, 1.7), (1.1, 1.7)], Vb, vmax)
    source(a0, (0.7, 0.2), (0.7, 3.2), Vcc / 2, vmax, "dc", f"{Vcc:.0f}V")
    source(a0, (1.1, 0.2), (1.1, 1.7), Vb / 2, vmax, "dc", f"{Vb:.2f}V")
    node_dot(a0, (2.3, 2.4), Vce, vmax)
    a0.text(2.65, 2.4, "out", color=FG, fontsize=8)
    charge_dots(a0, seg((2.3, 3.2), (2.3, 1.7 + 0.53), 24), Ic * 3e4,
                spacing=0.24, ms=3.6)
    a0.set_title(f"{'with' if Re>0 else 'without'} emitter degeneration",
                 fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[:, 1]), ORANGE)
    for vb_ in np.arange(0.58, 0.74, 0.02):
        a1.plot(vces, bjt_ic(vb_, vces) * 1e3, color=ORANGE, lw=0.8, alpha=0.45)
        a1.text(Vcc * 1.02, bjt_ic(vb_, Vcc) * 1e3, f"{vb_:.2f}", color=MUTED,
                fontsize=6.5, va="center")
    a1.plot([0, Vcc], [Vcc / Rc * 1e3, 0], color=POS, lw=2.0,
            label=f"load line, Rc = {Rc_k:.1f} kΩ")
    a1.plot([Vce0], [Ic0 * 1e3], "o", ms=9, color=DOT, label="20 °C")
    if abs(dT) > 0.1:
        a1.plot([Vce], [Ic * 1e3], "X", ms=11, color=NEG,
                label=f"+{dT:.0f} °C")
    a1.axvspan(0, 0.3, color=NEG, alpha=0.12)
    a1.text(0.15, Vcc / Rc * 1e3 * 0.5, "saturation", color=NEG, fontsize=7.5,
            rotation=90, va="center")
    a1.set_xlim(0, Vcc * 1.12); a1.set_ylim(0, Vcc / Rc * 1e3 * 1.25)
    a1.set_xlabel("$V_{CE}$  (V)"); a1.set_ylabel("$I_C$  (mA)")
    a1.legend(fontsize=7)
    swing = min(Vce - 0.3, Vcc - Vce)
    a1.set_title(f"the operating point is the intersection — "
                 f"undistorted swing ±{max(swing,0):.2f} V")

    drift = Ic / max(Ic0, 1e-15)
    readout(fig, 0.845, 0.90, [
        "SUPPLY", "─" * 26,
        f"Vcc         {Vcc:>10.1f}V",
        f"Rc          {Rc_k:>10.2f}kΩ",
        f"Re          {Re:>10.1f}Ω",
        f"Vbias       {Vb:>10.3f}V",
        "", "LOAD LINE", "─" * 26,
        f"Ic max      {Vcc/Rc*1e3:>10.4f}mA",
        f"Vce max     {Vcc:>10.2f}V",
        "", "AT 20 °C", "─" * 26,
        f"Ic          {Ic0*1e3:>10.5f}mA",
        f"Vce         {Vce0:>10.4f}V",
        f"headroom up {Vcc-Vce0:>10.3f}V",
        f"headroom dn {Vce0-0.3:>10.3f}V",
        f"P quiescent {Ic0*Vce0*1e3:>10.3f}mW",
        "", f"AT +{dT:.0f} °C", "─" * 26,
        f"Ic          {Ic*1e3:>10.5f}mA",
        f"drift       {drift:>10.3f}×",
        f"Vce         {Vce:>10.4f}V",
        "SATURATED" if Vce < 0.35 else "still active",
    ], color=NEG if (drift > 2 or Vce < 0.35) else GREEN)
    footer(fig, f"Vbe drifts −2 mV/°C   ·   without Re a 30 °C rise multiplies Ic "
                f"by ~10   ·   with Re it is ~1.05")
    plt.show()


w4 = dict(Vcc=widgets.FloatSlider(value=10, min=3, max=20, step=1,
                                  description="Vcc (V):", **SL),
          Rc_k=widgets.FloatSlider(value=4.7, min=0.5, max=20, step=0.1,
                                   description="Rc (kΩ):", **SL),
          Re=widgets.FloatSlider(value=0, min=0, max=2000, step=10,
                                 description="Re (Ω):", **SL),
          Vb=widgets.FloatSlider(value=0.66, min=0.55, max=3.0, step=0.005,
                                 description="bias V:", **SL),
          dT=widgets.FloatSlider(value=0, min=0, max=60, step=5,
                                 description="ΔT (°C):", **SL))
display(widgets.VBox([widgets.HBox([w4["Vcc"], w4["Rc_k"], w4["Re"]]),
                      widgets.HBox([w4["Vb"], w4["dT"]])]),
        widgets.interactive_output(draw_loadline, w4))

## The small-signal model — a straight line through one point

Nothing above is linear, so none of the analysis from the circuits notebooks applies. The standard escape is to pick an operating point, take the derivative there, and treat the device as linear for signals small enough that the curve has not bent yet:

$$i_c\approx g_m v_{be},\qquad g_m=\left.\frac{\partial I_C}{\partial V_{BE}}\right|_{Q}=\frac{I_C}{V_T}$$

That turns the transistor into the **hybrid-π** model — a resistor $r_\pi=\beta/g_m$, a controlled source $g_mv_{be}$, and an output resistance $r_o=V_A/I_C$ — and once it does, superposition, Thévenin and phasors all come back.

The price is a range limit, and the panel measures it rather than asserting it. For a BJT the linearization error is 1.9% at 1 mV of input swing, 9.4% at 5 mV, and **18.1% at 10 mV** — so the familiar "keep it under 10 mV" rule is really a *10% distortion* rule, and it is 5 mV that buys 10%.

The MOSFET is far more forgiving: a square law is a much better match to a straight line than an exponential is, and 10 mV costs only 0.5%. It takes **200 mV** of swing to reach the same 9% error the BJT hits at 5 mV. That is one of the reasons MOS circuits tolerate large signals gracefully while bipolar stages distort early — and why the distortion a BJT does produce is dominated by low-order harmonics you can predict.

In [ ]:
def draw_smallsignal(k, Ic_mA, swing_mV, device):
    Ic = Ic_mA * 1e-3
    A = swing_mV * 1e-3
    wt = 2 * np.pi * k / 120
    if device == "BJT":
        V0 = VT * np.log(Ic / IS_BJT)
        gm = Ic / VT
        vv = np.linspace(V0 - 4 * max(A, 0.002), V0 + 4 * max(A, 0.002), 400)
        curve = IS_BJT * np.exp(vv / VT)
        tang = Ic + gm * (vv - V0)
        vin = A * np.cos(wt)
        iex = IS_BJT * np.exp((V0 + vin) / VT) - Ic
        ilin = gm * vin
        unit_v, unit_i = "V", "mA"
    else:
        vov = np.sqrt(2 * Ic / KN)
        V0 = VTH_N + vov
        gm = KN * vov
        vv = np.linspace(V0 - 4 * max(A, 0.02), V0 + 4 * max(A, 0.02), 400)
        curve = 0.5 * KN * np.maximum(vv - VTH_N, 0) ** 2
        tang = Ic + gm * (vv - V0)
        vin = A * np.cos(wt)
        iex = 0.5 * KN * max(V0 + vin - VTH_N, 0) ** 2 - Ic
        ilin = gm * vin
        unit_v, unit_i = "V", "mA"
    t = np.linspace(0, 2, 400)
    sig = A * np.cos(2 * np.pi * t)
    if device == "BJT":
        ex_t = IS_BJT * np.exp((V0 + sig) / VT) - Ic
    else:
        ex_t = 0.5 * KN * np.maximum(V0 + sig - VTH_N, 0) ** 2 - Ic
    lin_t = gm * sig
    err = np.max(np.abs(ex_t - lin_t)) / max(np.max(np.abs(ex_t)), 1e-18) * 100
    vmax = 1.0

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.2, 0.55],
                          wspace=0.3, hspace=0.46, left=0.05, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = panel(fig.add_subplot(gs[:, 0]), BLUE)
    a0.plot(vv, curve * 1e3, color=POS, lw=1.8, label="the real device")
    a0.plot(vv, tang * 1e3, color=DOT, lw=1.3, ls="--",
            label=f"tangent, slope $g_m$")
    a0.plot([V0], [Ic * 1e3], "o", ms=9, color=FG, zorder=5)
    a0.axvspan(V0 - A, V0 + A, color=ORANGE, alpha=0.16)
    a0.plot([V0 + vin], [(Ic + iex) * 1e3], "o", ms=8, color=ORANGE, zorder=6)
    a0.set_xlabel(f"input voltage  ({unit_v})")
    a0.set_ylabel(f"output current  ({unit_i})")
    a0.legend(fontsize=7.5)
    a0.set_title(f"{device} — the model is this tangent, nothing more")

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    a1.plot(t, ex_t * 1e3, color=POS, lw=1.6, label="actual")
    a1.plot(t, lin_t * 1e3, color=DOT, lw=1.2, ls="--", label="linear model")
    a1.axvline((wt / (2 * np.pi)) % 2, color=FG, lw=1.0, ls=":")
    a1.set_xlabel("cycles"); a1.set_ylabel("Δi  (mA)")
    a1.legend(fontsize=7)
    a1.set_title(f"peak error {err:.2f}% at ±{swing_mV:.1f} mV")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    sw = np.logspace(np.log10(0.3), np.log10(400), 200)
    eb, em = [], []
    V0b = VT * np.log(Ic / IS_BJT)
    vovm = np.sqrt(2 * Ic / KN)
    for s in sw:
        a_ = s * 1e-3
        x = IS_BJT * np.exp((V0b + a_) / VT) - Ic
        eb.append(abs(x - (Ic / VT) * a_) / abs(x) * 100)
        y = 0.5 * KN * (vovm + a_) ** 2 - Ic
        em.append(abs(y - KN * vovm * a_) / abs(y) * 100)
    a2.semilogx(sw, eb, color=POS, lw=1.6, label="BJT")
    a2.semilogx(sw, em, color=ORANGE, lw=1.6, label="MOSFET")
    a2.axhline(10, color=MUTED, lw=0.8, ls=":")
    a2.axvline(swing_mV, color=FG, lw=1.0, ls="--")
    a2.set_ylim(0, 45); a2.set_xlabel("input swing  (mV peak)")
    a2.set_ylabel("error  (%)"); a2.legend(fontsize=7)
    a2.set_title("10% error at 5 mV for a BJT, 200 mV for a MOSFET")

    readout(fig, 0.845, 0.90, [
        "OPERATING POINT", "─" * 26,
        f"device      {device:>14s}",
        f"bias I      {Ic_mA:>10.4f}mA",
        f"bias V      {V0:>10.4f}V",
        "", "SMALL-SIGNAL", "─" * 26,
        f"gm          {gm*1e3:>10.4f}mS",
        (f"rpi = β/gm  {BETA/gm/1e3:>10.3f}kΩ" if device == "BJT"
         else f"rgs         {np.inf:>10.1f}Ω"),
        f"ro = VA/I   {VA/Ic/1e3:>10.3f}kΩ",
        f"gm·ro       {gm*VA/Ic:>10.1f}",
        "", "VALIDITY", "─" * 26,
        f"swing       {swing_mV:>10.2f}mV",
        f"peak error  {err:>10.2f}%",
        f"Δi linear   {gm*A*1e3:>10.5f}mA",
        f"Δi actual   {np.max(np.abs(ex_t))*1e3:>10.5f}mA",
        "", "BJT reference", "─" * 26,
        " 1 mV →  1.9%",
        " 5 mV →  9.4%",
        "10 mV → 18.1%",
        "", "MOS reference", "─" * 26,
        " 10 mV →  0.5%",
        "200 mV →  9.1%",
    ], color=GREEN if err < 10 else ORANGE)
    footer(fig, f"gm = ∂I/∂V at the bias point   ·   "
                f"the hybrid-π model is a tangent line, valid only near Q")
    plt.show()


_p5, _s5 = timeline(119, step=2)
w5 = dict(Ic_mA=widgets.FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05,
                                    description="bias I (mA):", **SL),
          swing_mV=widgets.FloatSlider(value=5, min=0.5, max=200, step=0.5,
                                       description="swing (mV):", **SL),
          device=widgets.Dropdown(options=["BJT", "MOSFET"], value="BJT",
                                  description="device:", **SL),
          k=_s5)
display(widgets.VBox([widgets.HBox([w5["Ic_mA"], w5["swing_mV"], w5["device"]]),
                      widgets.HBox([_p5, _s5])]),
        widgets.interactive_output(draw_smallsignal, w5))

## The current mirror — the first circuit that is only possible with matching

Two identical transistors sharing a base–emitter voltage carry the same current. That is the whole idea, and it is the most-used circuit in analog integrated design:

$$I_{ref}=\frac{V_{CC}-V_{BE}}{R},\qquad I_{out}\approx I_{ref}$$

The left transistor is diode-connected, so it *finds* whatever $V_{BE}$ its current needs. The right one is handed that same voltage and, being identical, produces the same current — without anyone measuring or setting it. This is why mirrors belong on a chip and not on a breadboard: the whole thing rests on the two devices being the same, which is free on silicon and hopeless with parts from a drawer.

Three imperfections show up in the panels, and each is exactly isolable.

**Base current.** Both bases are fed from the reference branch, so $I_{ref}=I_{out}+2I_B$ and the output falls short by exactly $1/(1+2\beta^{-1})$ — measured $0.98684$ at $\beta=150$ and $0.93750$ at $\beta=30$, matching the formula to five decimals.

**Early effect.** The two transistors only carry equal currents when they sit at equal $V_{CE}$. The reference is diode-connected at $V_{CE}=V_{BE}\approx0.65$ V while the output is wherever the load puts it, so the ratio is $(1+V_{out}/V_A)/(1+V_{BE}/V_A)$ — an error of $+8.5\%$ at $V_{out}=5$ V and $+28\%$ at 15 V, rather than the flat mirror the ideal picture suggests. Drag $V_{out}$ down to $0.7$ V and the ratio returns to $0.9997$.

**Mismatch**, which is the one that punishes the exponential. The ratio is $e^{\Delta V_{BE}/V_T}$ exactly, so 2 mV of offset between two supposedly identical devices is already **8.0%** of current error and 5 mV is **21.3%**. On a chip that offset is a few hundred microvolts; between two loose parts it can be tens of millivolts, which is why this circuit belongs on silicon.

In [ ]:
def mirror_solve(Vcc, R, Vout, dvbe_mV, beta):
    """Diode-connected reference and mirrored output.

    The reference transistor sits at Vce = Vbe, so its own Early factor is
    (1 + Vbe/VA); the output sits at Vout.  Both must be included or the
    mirror appears to have gain at zero mismatch.
    """
    vbe = 0.65
    for _ in range(300):
        iref = max((Vcc - vbe) / R, 1e-15)
        vbe = VT * np.log(iref / (IS_BJT * (1 + vbe / VA)))
    iref = (Vcc - vbe) / R
    isat = iref / (1 + vbe / VA)                       # de-embed the reference
    iout = isat * np.exp(dvbe_mV * 1e-3 / VT) * (1 + max(Vout, 0) / VA)
    iout_beta = iout / (1 + 2.0 / beta)                # base currents come from Iref
    return iref, iout_beta, vbe


def draw_mirror(Vcc, R_k, Vout, dvbe_mV, beta):
    R = R_k * 1e3
    iref, iout, vbe = mirror_solve(Vcc, R, Vout, dvbe_mV, beta)
    vo = np.linspace(0.2, Vcc, 300)
    io = np.array([mirror_solve(Vcc, R, v, dvbe_mV, beta)[1] for v in vo])
    vmax = Vcc

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1.25, 0.55],
                          wspace=0.3, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 4.4), (-0.6, 3.6))
    wire(a0, [(0.5, 3.2), (3.5, 3.2)], Vcc, vmax)
    resistor(a0, (1.3, 3.2), (1.3, 2.4), Vcc, vmax, f"R {R_k:.1f}k")
    transistor_npn(a0, (1.6, 1.6), 0.0, vmax, None)
    transistor_npn(a0, (3.4, 1.6), 0.0, vmax, None)
    wire(a0, [(1.3, 2.4), (1.3, 1.6 + 0.53)], vbe, vmax)
    wire(a0, [(1.3, 2.0), (0.95, 2.0), (0.95, 1.6)], vbe, vmax)
    wire(a0, [(0.95, 1.6), (1.6 - 0.63, 1.6)], vbe, vmax)
    wire(a0, [(0.95, 1.6), (3.4 - 0.63, 1.6)], vbe, vmax)
    wire(a0, [(1.6 + 0.29, 1.6 - 0.53), (1.6 + 0.29, 0.2)], 0.0, vmax)
    wire(a0, [(3.4 + 0.29, 1.6 - 0.53), (3.4 + 0.29, 0.2)], 0.0, vmax)
    wire(a0, [(0.5, 0.2), (3.7, 0.2)], 0.0, vmax)
    wire(a0, [(3.4 + 0.29, 1.6 + 0.53), (3.7, 2.6)], Vout, vmax)
    source(a0, (0.5, 0.2), (0.5, 3.2), Vcc / 2, vmax, "dc", f"{Vcc:.0f}V")
    node_dot(a0, (3.7, 2.6), Vout, vmax)
    a0.text(3.85, 2.75, f"Iout\n{iout*1e3:.4f} mA", color=DOT, fontsize=7)
    a0.text(1.55, 2.75, f"Iref\n{iref*1e3:.4f} mA", color=POS, fontsize=7)
    charge_dots(a0, seg((1.3, 3.2), (1.3, 2.15), 20), iref * 3e4, spacing=0.22,
                ms=3.4)
    charge_dots(a0, seg((3.7, 2.6), (3.4 + 0.29, 1.6 + 0.53), 20), iout * 3e4,
                spacing=0.22, ms=3.4)
    a0.set_title("both bases at the same voltage — so both carry the same current",
                 fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.plot(vo, io * 1e3, color=POS, lw=1.8)
    a1.axhline(iref * 1e3, color=MUTED, lw=1.0, ls="--")
    a1.text(vo[5], iref * 1e3 * 1.02, "Iref", color=MUTED, fontsize=7.5)
    a1.axvline(Vout, color=FG, lw=1.0, ls="--")
    a1.plot([Vout], [iout * 1e3], "o", ms=7, color=DOT)
    a1.set_ylim(0, max(io.max(), iref) * 1e3 * 1.25)
    a1.set_xlabel("output voltage  (V)"); a1.set_ylabel("$I_{out}$  (mA)")
    ro = VA / max(iout, 1e-15)
    a1.set_title(f"nearly a current source — $r_o$ = {ro/1e3:.1f} kΩ, not infinite")

    a2 = panel(fig.add_subplot(gs[1, 1]), NEG)
    dv = np.linspace(-15, 15, 300)
    a2.plot(dv, np.exp(dv * 1e-3 / VT), color=NEG, lw=1.8)
    a2.axhline(1.0, color=GRIDC, lw=0.8)
    a2.axvline(dvbe_mV, color=FG, lw=1.0, ls="--")
    a2.plot([dvbe_mV], [np.exp(dvbe_mV * 1e-3 / VT)], "o", ms=7, color=DOT)
    a2.set_xlabel("Vbe mismatch  (mV)"); a2.set_ylabel("current ratio")
    a2.set_title(f"5 mV of mismatch is {(np.exp(5e-3/VT)-1)*100:.0f}% of current error")

    readout(fig, 0.845, 0.90, [
        "REFERENCE", "─" * 26,
        f"Vcc         {Vcc:>10.1f}V",
        f"R           {R_k:>10.2f}kΩ",
        f"Vbe found   {vbe:>10.4f}V",
        f"Iref        {iref*1e3:>10.5f}mA",
        "", "OUTPUT", "─" * 26,
        f"Vout        {Vout:>10.2f}V",
        f"Iout        {iout*1e3:>10.5f}mA",
        f"ratio       {iout/max(iref,1e-15):>10.4f}",
        f"error       {(iout/max(iref,1e-15)-1)*100:>+10.2f}%",
        "", "ERROR SOURCES", "─" * 26,
        f"beta        {beta:>10.0f}",
        f"1/(1+2/β)   {1/(1+2/beta):>10.4f}",
        f"β error     {(1/(1+2/beta)-1)*100:>+10.2f}%",
        f"mismatch    {dvbe_mV:>+10.2f}mV",
        f"gives       {(np.exp(dvbe_mV*1e-3/VT)-1)*100:>+10.2f}%",
        f"Early ro    {VA/max(iout,1e-15)/1e3:>10.1f}kΩ",
        "", "matching is free on a",
        "chip, impossible with",
        "loose components",
    ], color=ORANGE if abs(dvbe_mV) > 2 else GREEN)
    footer(fig, f"Iout/Iref = exp(ΔVbe/VT)/(1+2/β)   ·   "
                f"5 mV mismatch → {(np.exp(5e-3/VT)-1)*100:.0f}% error   ·   "
                f"the exponential cuts both ways")
    plt.show()


w6 = dict(Vcc=widgets.FloatSlider(value=10, min=3, max=20, step=1,
                                  description="Vcc (V):", **SL),
          R_k=widgets.FloatSlider(value=9.3, min=1, max=50, step=0.1,
                                  description="R (kΩ):", **SL),
          Vout=widgets.FloatSlider(value=5, min=0.3, max=20, step=0.1,
                                   description="Vout (V):", **SL),
          dvbe_mV=widgets.FloatSlider(value=0, min=-15, max=15, step=0.5,
                                      description="mismatch (mV):", **SL),
          beta=widgets.FloatSlider(value=150, min=20, max=400, step=10,
                                   description="beta:", **SL))
display(widgets.VBox([widgets.HBox([w6["Vcc"], w6["R_k"], w6["Vout"]]),
                      widgets.HBox([w6["dvbe_mV"], w6["beta"]])]),
        widgets.interactive_output(draw_mirror, w6))